# TP 6 — LSTM Forecasting de rendement agricole avec **PyTorch**

*Notebook miroir de TP 5 (Keras) — même cas d'usage, même dataset, API différente.*

## Introduction

Ce notebook reproduit exactement le meme cas d'usage que TP 5 (LSTM Keras), en utilisant **PyTorch**.
La comparaison directe permet d'observer les differences concretes entre les deux frameworks :
- En PyTorch, la boucle d'entrainement est **explicite** (forward -> loss -> backward -> step)
- L'etat cache du LSTM `(h_n, c_n)` est manipule directement
- L'early stopping et la sauvegarde du modele sont implementes manuellement

**Objectifs pedagogiques :**
- Comprendre `nn.LSTM` et la gestion explicite de l'etat cache
- Ecrire une boucle d'entrainement PyTorch complete
- Implementer manuellement le validation loop et l'early stopping
- Comparer les API Keras vs PyTorch sur le meme probleme

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

# Reproductibilite
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', DEVICE)

## 1. Chargement et preparation des donnees (identique a TP 5)

In [ ]:
df = pd.read_excel('Data/crop_csv_file.xlsx')

# Agregation par annee
df_agg = df.groupby('Crop_Year')['Production'].sum().reset_index()
df_agg.columns = ['Year', 'Production']
df_agg = df_agg.sort_values('Year').reset_index(drop=True)

# Visualisation
plt.figure(figsize=(12, 4))
plt.plot(df_agg['Year'], df_agg['Production'], marker='o', linewidth=2, color='#3498db')
plt.title('Production agricole totale par annee (1997-2014)', fontsize=14)
plt.xlabel('Annee'); plt.ylabel('Production (tonnes)')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(df_agg)

## 2. Sliding Window et conversion en tenseurs PyTorch

Meme logique que TP 5, avec une difference cle :
PyTorch utilise `torch.FloatTensor` et le DataLoader gere les mini-batches.

In [ ]:
def create_sequences(series, window_size):
    """Cree des paires (X, y) pour apprentissage supervise sur serie temporelle."""
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i:i + window_size])
        y.append(series[i + window_size])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Normalisation
scaler = MinMaxScaler()
production_scaled = scaler.fit_transform(df_agg[['Production']]).flatten().astype(np.float32)

WINDOW_SIZE = 3
X_np, y_np = create_sequences(production_scaled, WINDOW_SIZE)

# Reshape : (samples, timesteps, features)
X_np = X_np.reshape(X_np.shape[0], X_np.shape[1], 1)
y_np = y_np.reshape(-1, 1)

# Split chronologique 80/20
split = int(len(X_np) * 0.8)
X_train_np, X_test_np = X_np[:split], X_np[split:]
y_train_np, y_test_np = y_np[:split], y_np[split:]

# Conversion en tenseurs PyTorch
X_train_t = torch.from_numpy(X_train_np).to(DEVICE)
y_train_t = torch.from_numpy(y_train_np).to(DEVICE)
X_test_t  = torch.from_numpy(X_test_np).to(DEVICE)
y_test_t  = torch.from_numpy(y_test_np).to(DEVICE)

# DataLoader pour mini-batch training
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=4, shuffle=False)

print('X_train shape:', X_train_t.shape, '| y_train shape:', y_train_t.shape)
print('X_test  shape:', X_test_t.shape,  '| y_test  shape:', y_test_t.shape)

## 3. Definition du modele LSTM (PyTorch)

**Difference cle vs Keras :**
- En Keras : `LSTM(64, return_sequences=True)` gere l'etat cache de facon transparente
- En PyTorch : `nn.LSTM(...)` retourne explicitement `(output, (h_n, c_n))`
  — on doit extraire `h_n[-1]` (dernier etat cache) pour la couche Dense

Architecture : identique a TP 5 — LSTM(64) -> Dropout -> LSTM(32) -> Linear(1)

In [ ]:
class LSTMForecaster(nn.Module):
    """LSTM pour le forecasting de production agricole.
    
    Equivalent PyTorch du modele Keras defini dans TP 5.
    Difference principale : l'etat cache (h_n, c_n) est manipule explicitement.
    """
    def __init__(self, input_size=1, hidden_size_1=64, hidden_size_2=32, dropout=0.2):
        super(LSTMForecaster, self).__init__()
        # Couche LSTM 1 : equivalent de LSTM(64, return_sequences=True)
        self.lstm1 = nn.LSTM(input_size, hidden_size_1, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        # Couche LSTM 2 : equivalent de LSTM(32)
        self.lstm2 = nn.LSTM(hidden_size_1, hidden_size_2, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        # Couche de sortie
        self.fc = nn.Linear(hidden_size_2, 1)

    def forward(self, x):
        # LSTM 1 — retourne toute la sequence
        out1, _ = self.lstm1(x)          # out1: (batch, seq_len, hidden_1)
        out1 = self.dropout1(out1)
        # LSTM 2 — retourne (output, (h_n, c_n))
        out2, (h_n, _) = self.lstm2(out1) # h_n: (1, batch, hidden_2)
        out2 = self.dropout2(out2)
        # On extrait le dernier etat cache (equivalent de return_sequences=False)
        last_hidden = h_n.squeeze(0)     # (batch, hidden_2)
        return self.fc(last_hidden)      # (batch, 1)


model = LSTMForecaster().to(DEVICE)
print(model)
print('\nNombre de parametres:',
      sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Boucle d'entrainement PyTorch (explicite)

**Difference cle vs Keras :** au lieu de `model.fit(...)`, on ecrit explicitement :
1. `optimizer.zero_grad()` — remet les gradients a zero
2. `output = model(X)` — forward pass
3. `loss = criterion(output, y)` — calcul de la loss
4. `loss.backward()` — retropropagation du gradient
5. `optimizer.step()` — mise a jour des poids

L'early stopping et la sauvegarde sont aussi implémentes manuellement.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Parametres d'early stopping
EPOCHS     = 200
PATIENCE   = 20
best_val   = float('inf')
no_improve = 0
best_weights = None

train_losses, val_losses = [], []

# Split validation manuel (20% des donnees train)
val_split = int(len(X_train_t) * 0.8)
X_tr = X_train_t[:val_split]; y_tr = y_train_t[:val_split]
X_val = X_train_t[val_split:]; y_val = y_train_t[val_split:]

for epoch in range(EPOCHS):
    # ---- Phase entrainement ----
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in DataLoader(TensorDataset(X_tr, y_tr), batch_size=4, shuffle=False):
        optimizer.zero_grad()             # 1. Remise a zero des gradients
        output = model(X_batch)           # 2. Forward pass
        loss = criterion(output, y_batch) # 3. Calcul de la loss
        loss.backward()                   # 4. Retroproagation
        optimizer.step()                  # 5. Mise a jour des poids
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(X_tr))

    # ---- Phase validation ----
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = criterion(val_pred, y_val).item()
    val_losses.append(val_loss)

    # ---- Early stopping manuel ----
    if val_loss < best_val:
        best_val = val_loss
        no_improve = 0
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print('Early stopping a l\'epoch', epoch + 1)
            break

    if (epoch + 1) % 20 == 0:
        print('Epoch {:3d} | Train MSE: {:.6f} | Val MSE: {:.6f}'.format(
              epoch + 1, train_losses[-1], val_losses[-1]))

# Restaurer les meilleurs poids
model.load_state_dict(best_weights)
torch.save(best_weights, 'best_lstm_pytorch.pt')
print('Meilleur modele sauvegarde dans best_lstm_pytorch.pt')

In [ ]:
# Courbes de loss
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train_losses, label='Train Loss (MSE)', color='#3498db', linewidth=2)
ax.plot(val_losses, label='Val Loss (MSE)', color='#e74c3c', linewidth=2)
ax.set_title('Courbes de loss — LSTM PyTorch', fontsize=14)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Nombre d\'epochs effectuees:', len(train_losses))

## 5. Evaluation et visualisation des predictions

In [ ]:
# Predictions sur le jeu de test
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t).cpu().numpy().flatten()

# Denormalisation
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = scaler.inverse_transform(y_test_np)

# Metriques
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)
print('MAE  : {:,.0f} tonnes'.format(mae))
print('RMSE : {:,.0f} tonnes'.format(rmse))

# Visualisation
test_years  = df_agg['Year'].values[WINDOW_SIZE + split:]
train_years = df_agg['Year'].values[WINDOW_SIZE:WINDOW_SIZE + split]
y_train_orig = scaler.inverse_transform(y_train_np).flatten()

plt.figure(figsize=(12, 5))
plt.plot(train_years, y_train_orig, label='Train', color='#95a5a6', linewidth=1.5)
plt.plot(test_years, y_true, label='Realite (test)', color='#3498db', linewidth=2, marker='o')
plt.plot(test_years, y_pred, label='Prediction LSTM PyTorch',
         color='#e67e22', linewidth=2, marker='s', linestyle='--')
plt.title('Forecasting de production agricole — LSTM PyTorch', fontsize=14)
plt.xlabel('Annee'); plt.ylabel('Production (tonnes)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Resume comparatif PyTorch vs Keras

| Aspect | PyTorch (ce notebook) | Keras (TP 5) |
|---|---|---|
| **Definition LSTM** | `nn.LSTM(1, 64, batch_first=True)` | `LSTM(64, return_sequences=True)` |
| **Etat cache** | `(output, (h_n, c_n))` — manipulation explicite | Gere automatiquement |
| **Boucle entrainement** | Manuelle : `zero_grad -> forward -> backward -> step` | `model.fit(callbacks=[...])` |
| **Early stopping** | Condition `if val_loss < best_val: ...` a coder | `EarlyStopping` callback integre |
| **Sauvegarde** | `torch.save(model.state_dict(), path)` | `ModelCheckpoint` callback |
| **Avantage** | Debuggabilite totale, custom gradient | Rapidite de prototypage |